In [1]:
%matplotlib inline
# Cell 1 — WO-02 Step 3, family 5: Distribution. Load fixtures and population payloads (read-only; the API is never called)
import sys
from pathlib import Path
import pandas as pd

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'scripts' / 'edop').exists())
sys.path.insert(0, str(ROOT))
from scripts.edop.dashboard import wo02_payloads as P, wo02_rules_distribution as D, wo02_analysis as A

fx, pop, prov = P.load_all()
W = P.population_weights(pop)
FX_BY_ID = {q['id']: q for q in fx}
POP_BY_ID = {q['id']: q for q in pop}
print('provenance:', prov)
print(f'fixtures with a 200: {len(fx)} | population queries with a 200: {len(pop)}')


provenance: {'app_version': '0.4', 'git_commit': 'bb80fee', 'git_describe': 'v0.4.2-12-gbb80fee', 'branch': 'dashboard', 'planned': '2026-10-30', 'captured': '2026-10-30'}
fixtures with a 200: 33 | population queries with a 200: 505


In [2]:
# Cell 2 — The six Distribution statement types (rules in scripts/edop/dashboard/wo02_rules_distribution.py)
for s in D.STATEMENTS:
    print(f"{s['id']}  {s['name']}")
    print(f"      applies to: {s['applies_to']}")
    print(f"      template  : {s['template']}")
print()
print(f"the engine clears the coherence label below {D.MIN_COHERENT} basins")


DIS-01  Core window and core share
      applies to: buffer, polity; area_weighted, distribution_only; L6, L8
      template  : {name}: {share} of the area lies between the {ord} and {ord} percentiles.
DIS-02  Middle 80% of the area (p10 to p90)
      applies to: buffer, polity; area_weighted, distribution_only; L6, L8
      template  : {name}: the middle 80% of the area (by weight) lies between the {ord} and {ord} percentiles.
DIS-03  Lowest and highest basin
      applies to: buffer, polity; area_weighted; L6, L8
      template  : {name}: across the {n} basins, values run from the {ord} to the {ord} percentile.
DIS-04  Split: one value carries much of the area, a separate group lies elsewhere
      applies to: buffer, polity; area_weighted, distribution_only; L6, L8
      template  : {name}: at least half of the area is tied at one value inside the {ord} to {ord} percentile window, and {share} of the area lies outside it; the headline, the {ord} percentile, is the mean over both.
DIS

In [3]:
# Cell 3 — Fixtures: apply every rule to every Band A-E row; count where each applies and why not
fdf = A.apply_fixtures(fx, D)
print(A.fixture_table(fdf).to_string())
print()
print('why a statement did not apply (count of rows):')
print(A.reason_table(fdf).to_string())


res                  applied  not_applicable
id     scope                                
DIS-01 buffer            111             149
       polity            146              62
       single_basin        0            1196
DIS-02 buffer            111             149
       polity            146              62
       single_basin        0            1196
DIS-03 buffer            106             154
       polity            134              74
       single_basin        0            1196
DIS-04 buffer              5             255
       polity             15             193
       single_basin        0            1196
DIS-05 buffer              4             256
       polity              0             208
       single_basin        0            1196
DIS-06 buffer            106             154
       polity            131              77
       single_basin        0            1196

why a statement did not apply (count of rows):
id      reason                      
DIS-01  fewer_t

In [4]:
# Cell 4 — Draft sentences on chosen queries (up to 2 per statement type), to read and judge
for qid in ['F03', 'F04', 'F18', 'F22']:
    g = fdf[(fdf.qid == qid) & fdf.applies]
    print(f"--- {qid}: {FX_BY_ID[qid]['condition']}")
    for sid, gg in g.groupby('id'):
        for _, r in gg.head(2).iterrows():
            print(f"  {sid}  {r['text']}")
    print()
print('--- P006b (population): a buffer with split rows')
for r in D.evaluate_query(POP_BY_ID['P006b']):
    if r['applies'] and r['id'] in ('DIS-04',):
        print(f"  {r['id']}  {r['variable']}: {r['text']}")


--- F03: small buffer (fewer than 5 basins)
  DIS-03  Groundwater table depth: across the 4 basins, values run from the 14th to the 50th percentile.
  DIS-03  River area: across the 4 basins, values run from the 68th to the 77th percentile.
  DIS-05  Groundwater table depth: this buffer has only 4 basins, so no spread label is given (it needs at least 5).
  DIS-05  River area: this buffer has only 4 basins, so no spread label is given (it needs at least 5).

--- F04: large buffer; suppressed scores; section 2.6 reference counts
  DIS-01  Aridity index (P/PET): 100% of the area lies between the 0th and 30th percentiles.
  DIS-01  Elevation maximum: 98% of the area lies between the 4th and 34th percentiles.
  DIS-02  Aridity index (P/PET): the middle 80% of the area (by weight) lies between the 6th and 13th percentiles.
  DIS-02  Elevation maximum: the middle 80% of the area (by weight) lies between the 16th and 32nd percentiles.
  DIS-03  Aridity index (P/PET): across the 9 basins, valu

In [5]:
# Cell 5 — Population sample: how often each statement applies (share of all Band A-E rows), weighted to the world and not
pdf = A.apply_population(pop, W, D)
print('rows evaluated per statement type:', len(pdf) // len(D.STATEMENTS))
print('WEIGHTED by climate-zone size (percent of rows):')
print((A.rates(pdf) * 100).round(1).to_string())
print()
print('UNWEIGHTED (percent of rows):')
print((A.unweighted_rates(pdf) * 100).round(1).to_string())
print()
print('polities are not in the population sample (5 queries, all fixtures); see Cell 3')


rows evaluated per statement type: 26260
WEIGHTED by climate-zone size (percent of rows):
kind    buffer L6  single L6  single L8
id                                     
DIS-01       48.0        0.0        0.0
DIS-02       48.0        0.0        0.0
DIS-03       45.2        0.0        0.0
DIS-04        2.1        0.0        0.0
DIS-05        1.6        0.0        0.0
DIS-06       45.9        0.0        0.0

UNWEIGHTED (percent of rows):
kind    buffer L6  single L6  single L8
id                                     
DIS-01       45.1        0.0        0.0
DIS-02       45.1        0.0        0.0
DIS-03       42.7        0.0        0.0
DIS-04        2.1        0.0        0.0
DIS-05        1.9        0.0        0.0
DIS-06       43.0        0.0        0.0

polities are not in the population sample (5 queries, all fixtures); see Cell 3


In [6]:
# Cell 6 — The numbers inside the statements (weighted): core share, width of the middle 80%, lowest-to-highest range
print('DIS-01 core share; DIS-02 width of the middle 80% (p90 - p10); DIS-03 lowest-to-highest range; DIS-04 share outside the window:')
print(A.stat_quantiles(pdf, ['DIS-01', 'DIS-02', 'DIS-03', 'DIS-04']).round(2).to_string(index=False))
print()
lab = pdf[pdf.applies & pdf.id.isin(['DIS-04', 'DIS-06'])].copy()
lab['label'] = lab.coherence
num = lab.groupby(['kind', 'label']).w.sum()
den = lab.groupby('kind').w.sum()
share = num / num.index.get_level_values('kind').map(den).values
print('weighted share of rows with 5+ basins, ok and not suppressed, by coherence label (percent):')
print((share * 100).round(1).to_string())
n5 = pdf[(pdf.id == 'DIS-05') & pdf.applies]
print()
print('DIS-05 (2 to 4 basins): rows by basin count:', n5.n_units.value_counts().sort_index().to_dict())


DIS-01 core share; DIS-02 width of the middle 80% (p90 - p10); DIS-03 lowest-to-highest range; DIS-04 share outside the window:
    id      kind    n  p10   p25   p50   p75   p90
DIS-01 buffer L6 2323 0.74  0.90  1.00  1.00  1.00
DIS-02 buffer L6 2323 2.08  6.43 13.95 28.20 49.13
DIS-03 buffer L6 2198 5.94 12.73 25.97 45.47 69.45
DIS-04 buffer L6  107 0.12  0.13  0.15  0.21  0.34

weighted share of rows with 5+ basins, ok and not suppressed, by coherence label (percent):
kind       label       
buffer L6  concentrated    90.3
           split            4.3
           spread           5.4

DIS-05 (2 to 4 basins): rows by basin count: {2: 5, 4: 95}


In [7]:
# Cell 7 — Sweep: the engine's concentrated/spread label rests on a core-share cutoff of 0.65 (provisional). Evaluate a grid; do not choose
CUTS = [0.5, 0.55, 0.6, 0.65, 0.7, 0.75, 0.8]
sw = A.cutoff_sweep(pdf, 'DIS-06', CUTS)
print('share of labelled rows (concentrated or spread) that would read "concentrated" at each core-share cutoff, and share whose wording changes between 0.5 and 0.8:')
print(sw.round(3).to_string(index=False))
print()
print('variables whose label flips (count of rows with a core share from 0.5 up to 0.8):')
print(A.flipping_variables(pdf, 'DIS-06', CUTS[0], CUTS[-1]).to_string())
print()
print('the split rule (a tied score holding at least half of the weight) cannot be swept: the tied weight is not in the payload')


share of labelled rows (concentrated or spread) that would read "concentrated" at each core-share cutoff, and share whose wording changes between 0.5 and 0.8:
     kind    n  >= 0.5  >= 0.55  >= 0.6  >= 0.65  >= 0.7  >= 0.75  >= 0.8  flips
buffer L6 2216   0.993     0.98   0.966    0.944    0.92    0.895   0.863   0.13

variables whose label flips (count of rows with a core share from 0.5 up to 0.8):
variable
river_area                  35
slope_upstream              15
cropland_extent_upstream    11
pasture_extent              11
wet_pct_grp2                10
wet_pct_grp1                10
wet_pct_grp1_upstream        9
elev_max                     9

the split rule (a tied score holding at least half of the weight) cannot be swept: the tied weight is not in the payload


In [8]:
# Cell 8 — Checks: does the engine's label follow core share >= 0.65, and what a split row looks like
print('label versus core_share >= 0.65 (eligible rows: 5+ basins, ok, not suppressed):', dict(D.label_consistency(pop + fx)))
sc = pd.DataFrame(D.split_checks(pop + fx), columns=['qid', 'variable', 'score', 'in_window', 'd_p90', 'd_p10', 'core'])
print()
print(f'split rows: {len(sc)} | headline score lies inside the core window: {int(sc.in_window.sum())}')
print('distance from the headline to p90 and to p10 (median):', round(sc.d_p90.median(), 1), round(sc.d_p10.median(), 1), '-> the headline is neither the tied value nor an edge')
print('core share of split rows: min', round(sc.core.min(), 2), 'median', round(sc.core.median(), 2), 'max', round(sc.core.max(), 2))
print('variables:')
print(sc.variable.value_counts().head(8).to_string())
print()
d = pdf[(pdf.id == 'DIS-01')]
print('rows by reason DIS-01 did not apply (population, count):')
print(d[~d.applies].reason.value_counts().to_string())


label versus core_share >= 0.65 (eligible rows: 5+ basins, ok, not suppressed): {('concentrated', 'core>=cut'): 2295, ('spread', 'core<cut'): 158, ('split', 'core>=cut'): 120, ('split', 'core<cut'): 7}

split rows: 127 | headline score lies inside the core window: 119
distance from the headline to p90 and to p10 (median): 16.5 16.7 -> the headline is neither the tied value nor an edge
core share of split rows: min 0.57 median 0.84 max 0.9
variables:
variable
wet_pct_grp1             17
wet_pct_grp2_upstream    12
wet_pct_grp1_upstream    10
karst                     9
wet_pct_grp2              9
karst_upstream            9
pct_sand_upstream         6
reservoir_vol             6

rows by reason DIS-01 did not apply (population, count):
reason
one_unit                        15466
method:class_mixture             5050
method:dominant_basin            1515
suppressed                        773
method:flag_fraction              505
status:outside_active_domain      423
status:no_data      

In [9]:
# Cell 9 — Write this family's rows to the master table (replaces any earlier Distribution rows)
fxt, rw, ru = A.fixture_table(fdf), A.rates(pdf), A.unweighted_rates(pdf)
s0 = sw.iloc[0]
rows = []
for s in D.STATEMENTS:
    sid = s['id']
    f = fxt.loc[sid]
    row = dict(s, family='Distribution')
    row['fixture_result'] = f"{int(f.applied.sum())} rows applied, {int(f.not_applicable.sum())} not applicable ({len(fx)} fixtures)"
    row['population_result'] = ('share of all A-E rows, weighted: ' + ', '.join(f"{k} {rw.loc[sid, k]:.1%}" for k in rw.columns)
                                + '; unweighted: ' + ', '.join(f"{k} {ru.loc[sid, k]:.1%}" for k in ru.columns) + '; polities not sampled')
    if sid == 'DIS-06':
        row['sweep_result'] = (f"label flips: share reading 'concentrated' is {s0['>= 0.5']:.0%} at a core-share cutoff of 0.5 and {s0['>= 0.8']:.0%} at 0.8; "
                               f"{s0['flips']:.0%} of labelled rows change wording between 0.5 and 0.8; DIS-01 states the core share as a number")
    elif sid == 'DIS-04':
        row['sweep_result'] = 'not sweepable: the tied weight is not in the payload'
    else:
        row['sweep_result'] = 'no cutoff'
    rows.append(row)
n_rows = P.upsert_master('Distribution', rows)
print('master table now has', n_rows, 'rows ->', P.MASTER)


master table now has 28 rows -> /Users/karlg/Documents/repos/_edops/notebooks/edop/dashboard/wo02_master_table.csv
